In [2]:
import faiss
import numpy as np
from elasticsearch import Elasticsearch
from transformers import AutoTokenizer, AutoModelForCausalLM, BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer
import chromadb
import os


In [5]:
from huggingface_hub import login
import os

hf_api_token = os.getenv('HF_token')
login(token=hf_api_token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  

In [8]:
from langchain_groq import ChatGroq

import os
from dotenv import load_dotenv

load_dotenv()

def get_llm():
    """
    Returns the language model instance.

    This function initializes and returns a ChatGroq language model configured with the specified model name,
    temperature, maximum tokens, and other settings.

    Returns:
        ChatGroq: An instance of the ChatGroq language model.
    """
    llm = ChatGroq(
        model="mixtral-8x7b-32768",
        temperature=0,
        max_tokens=1024,
    )
    return llm

# ElasticSearch for Textual Retrieval

In [9]:
embeddings = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dorot\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
from elasticsearch import Elasticsearch, ConnectionError
from langchain_elasticsearch import ElasticsearchStore

In [15]:
print(f"Type of es: {type(es)}")
print(f"ES Target: {es}")

Type of es: <class 'elasticsearch.Elasticsearch'>
ES Target: <Elasticsearch(['https://2e8be0fdffac440795fcfbecf86079b4.us-central1.gcp.cloud.es.io:443'])>


In [20]:
from elasticsearch import Elasticsearch

# es = Elasticsearch(
#     ['https://2e8be0fdffac440795fcfbecf86079b4.us-central1.gcp.cloud.es.io'],
#     bearer_auth=('elastic', 'aPRfFBGj1FkOMgvm7XqkgAFJ'),
#     )
es = Elasticsearch(hosts=['https://2e8be0fdffac440795fcfbecf86079b4.us-central1.gcp.cloud.es.io'],  http_auth=('elastic', 'aPRfFBGj1FkOMgvm7XqkgAFJ'))

es.ping()

C:\Users\dorot\AppData\Local\Temp\ipykernel_317532\2263511075.py:7: DeprecationWarning: The 'http_auth' parameter is deprecated. Use 'basic_auth' or 'bearer_auth' parameters instead
  es = Elasticsearch(hosts=['https://2e8be0fdffac440795fcfbecf86079b4.us-central1.gcp.cloud.es.io'],  http_auth=('elastic', 'aPRfFBGj1FkOMgvm7XqkgAFJ'))


True

In [22]:
# Define the mapping for the index
index_body = {
    "mappings": {
        "properties": {
            "content": {"type": "text"}
        }
    }
}

# Create an index named 'movies'
index_name = 'movies'
if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name, body=index_body)
    print(f"Index '{index_name}' created.")
else:
    print(f"Index '{index_name}' already exists.")


NotFoundError: NotFoundError(404, "{'ok': False, 'message': 'Unknown resource.'}")

# Load reranking model

In [23]:
# Load the tokenizer for the reranking model
rerank_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Load the BERT model for sequence classification (used for reranking)
rerank_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 4e8159cf-391b-495c-824a-0de000b7ad40)')' thrown while requesting HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [24]:
chat_groq_model = get_llm()

# Function for Advanced Query Transformation
### This function modifies and expands the input query for better retrieval results.

In [25]:
def advanced_query_transformation(query):
    """
    Transforms the input query by adding synonyms, extensions, or modifying the structure
    for better search performance.

    Args:
        query (str): The original query.

    Returns:
        str: The transformed query with added synonyms or related terms.
    """
    # Example transformation: adding an OR clause with a related term
    expanded_query = query + " OR related_term"
    return expanded_query

# Function for Advanced Query Routing


### This function decides the retrieval method (textual or vector-based) based on the query type.


In [26]:
def advanced_query_routing(query):
    """
    Determines the retrieval method based on the presence of specific keywords in the query.

    Args:
        query (str): The user's query.

    Returns:
        str: 'textual' if the query requires text-based retrieval, 'vector' otherwise.
    """
    if "specific_keyword" in query:
        return "textual"
    else:
        return "vector"

# Fusion Retrieval Function


### This function retrieves documents using both vector-based and textual retrieval methods.

In [ ]:
# Fusion Retrieval Function
def fusion_retrieval(query, top_k=5):
    """
    Retrieves the top_k most relevant documents using a combination of vector-based
    and textual retrieval methods.

    Args:
        query (str): The search query.
        top_k (int): The number of top documents to retrieve.

    Returns:
        list: A list of combined results from both vector and textual retrieval methods.
    """
    # Vector-based retrieval using sentence embeddings
    query_embedding = sentence_model.encode(query).tolist()
    vector_results = collection.query(query_embeddings=[query_embedding], n_results=min(top_k, len(documents)))

    # Textual retrieval using Elasticsearch
    es_body = {
        "size": top_k,  # Move size into body
        "query": {
            "match": {
                "content": query
            }
        }
    }
    es_results = es.search(index=index_name, body=es_body)
    es_documents = [hit["_source"]["content"] for hit in es_results['hits']['hits']]

    # Combine results from both retrieval methods
    combined_results = vector_results['documents'][0] + es_documents

    return combined_results


# Document Reranking Function


### This function reranks retrieved documents based on their relevance to the query.



In [ ]:
import torch.nn.functional as F

# Document Reranking Function
def rerank_documents(query, documents):
    """
    Reranks the retrieved documents based on their relevance to the query using a pre-trained
    BERT model.

    Args:
        query (str): The user's query.
        documents (list): A list of documents retrieved from the search.

    Returns:
        list: A list of reranked documents, sorted by relevance.
    """
    inputs = [rerank_tokenizer.encode_plus(query, doc, return_tensors='pt', truncation=True, padding=True) for doc in documents]

    # Use logits to get scores
    scores = []
    for input in inputs:
        outputs = rerank_model(**input)
        logits = outputs.logits
        probabilities = F.softmax(logits, dim=1)
        positive_class_probability = probabilities[:, 1].item()  # Assuming the second element represents the positive class
        scores.append(positive_class_probability)

    ranked_docs = sorted(zip(documents, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, score in ranked_docs]


# Context Selection and Compression


### This function selects and summarizes the context to be used in the final answer generation.

In [ ]:
summarizer = pipeline("summarization")

In [ ]:
def select_and_compress_context(documents):
    """
    Summarizes the content of the retrieved documents to create a compressed context.

    Args:
        documents (list): A list of documents to summarize.

    Returns:
        list: A list of summarized texts for each document.
    """
    summarized_context = []
    for doc in documents:
        input_length = len(doc.split())  # Calculate input length based on word count
        max_length = min(100, input_length)  # Set max_length to input_length if smaller than 100
        summary = summarizer(doc, max_length=max_length, min_length=5, do_sample=False)[0]['summary_text']
        summarized_context.append(summary)
    return summarized_context

# Answer Generation Function


### This function generates the final answer based on the provided context and query.

In [ ]:
# Answer Generation Function
def generate_answer(query, chunks, llm):
    """
    Generates an answer based on the input query and context chunks using a language model.

    Args:
        query (str): The user's query.
        chunks (list): A list of context chunks to inform the answer.
        llm (ChatGroq): An instance of the ChatGroq language model.

    Returns:
        str: The generated answer.
    """
    # Combine chunks into a single context string
    context = "\n\n".join(chunks)

    # Construct the prompt for the language model as a string
    prompt = f"""[INST]
Instruction: You're an expert in movie suggestions. Your task is to analyze carefully the context and come up with an exhaustive answer to the following question:
{query}

Here is the context to help you:

{context}

[/INST]"""

    # Invoke the language model with the prompt
    response = llm.invoke(prompt)  # Pass the prompt as a string directly

    # Since response is likely an AIMessage object, access the content directly
    generated_text = response.content

    return generated_text


# Full Advanced Retrieval-Augmented Generation (RAG) Pipeline


In [ ]:
def advanced_rag_pipeline(query):
    """
    The main pipeline function for the Advanced Retrieval-Augmented Generation (RAG) system.
    It processes the query, retrieves relevant documents, reranks them, selects and compresses
    the context, and finally generates an answer.

    Args:
        query (str): The user's input query.

    Returns:
        str: The final generated answer.
    """
    # Transform and route query
    transformed_query = advanced_query_transformation(query)
    retrieval_method = advanced_query_routing(transformed_query)

    # Retrieve documents using fusion retrieval
    retrieved_documents = fusion_retrieval(transformed_query)

    # Rerank documents based on relevance
    ranked_documents = rerank_documents(query, retrieved_documents)

    # Select and compress context for answer generation
    context = select_and_compress_context(ranked_documents)

    # Generate final answer based on the context
    final_answer = generate_answer(query, context, chat_groq_model)
    return final_answer

In [ ]:
import chromadb

# Initialize ChromaDB client and create collection
client = chromadb.Client()

# Define the collection name
collection_name = "movies"

try:
    # Attempt to create the collection in ChromaDB
    collection = client.create_collection(name=collection_name)
    print(f"Collection '{collection_name}' created successfully.")

    # Define the documents to be inserted into the collection
    documents = [
        {"id": "1", "content": "The Shawshank Redemption is a great movie to watch on a rainy day."},
        {"id": "2", "content": "Forrest Gump is an uplifting film perfect for a rainy afternoon."}
    ]

    # Extract the IDs and content for insertion
    ids = [doc["id"] for doc in documents]
    contents = [doc["content"] for doc in documents]

    # Insert documents into the collection
    collection.add(ids=ids, documents=contents)
    print("Documents inserted successfully.")

except Exception as e:
    print(f"Collection '{collection_name}' already exists. No need to create it again.")
    # Optionally, you could fetch the existing collection here
    collection = client.get_collection(name=collection_name)

except Exception as e:
    print(f"An error occurred: {e}")


In [ ]:
# Example query
query = "What are some good movies to watch on a rainy day?"

# Run the query through the Advanced RAG Pipeline
answer = advanced_rag_pipeline(query)

# Output the generated answer
print(answer)